# Interactive four-lepton OmniFold

Run **one iteration at a time**, inspect its three calibrated density ratios, and compare the unfolded MC with the independent Herwig truth. Rerunning the **Run the next iteration** cell appends one iteration to the same campaign. All models and completed iterations are stored on disk.

This notebook expects background-removal-decorated pseudo-data and the original `gg_H_pythia.root`. An optional **full** `gg_H_herwig.root` supplies fiducial truth closure, including reconstruction losses. It is never used to train the ratios.

Choose the **FourLepton OmniFold** kernel in Unity OnDemand. Installation and the statistical definitions are in [README.md](README.md). Use an interactive compute allocation with one GPU and suitable RAM. The setup cell reports whether CUDA is available.

Running all cells fits **one** potentially expensive iteration. To inspect an existing campaign without training, skip the next-iteration cell.

## Locate the checkout and inspect the kernel

The notebook can be launched from the repository root or its `omnifold` directory. If you copied it elsewhere, set `REPO_HINT` to the checkout's absolute path.

In [ ]:
from pathlib import Path
import html
import json
import os
import sys

REPO_HINT = None  # e.g. Path("/work/YOUR_GROUP/YOUR_USER/FourLeptonUnfolding")
search_roots = [Path(REPO_HINT).expanduser().resolve()] if REPO_HINT else [Path.cwd(), *Path.cwd().parents]
REPO = next((path for path in search_roots if (path / "BackgroundRemoval" / "common.py").is_file() and (path / "omnifold").is_dir()), None)
if REPO is None:
    raise FileNotFoundError("Set REPO_HINT to your FourLeptonUnfolding checkout.")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, Markdown, display
from omnifold import OmniFoldSession, RatioConfig

cpu_threads = max(1, int(os.environ.get("SLURM_CPUS_PER_TASK", "4")))
torch.set_num_threads(cpu_threads)
print("Repository:", REPO)
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CPU threads:", cpu_threads)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU mode. For GPU training, request a GPU in the Unity OnDemand allocation.")


def show_rows(rows):
    """A compact notebook table without requiring pandas."""
    rows = list(rows)
    if not rows:
        print("No completed iterations yet.")
        return
    keys = list(dict.fromkeys(key for row in rows for key in row))
    heading = "".join(f"<th>{html.escape(str(key))}</th>" for key in keys)
    body = "".join("<tr>" + "".join(f"<td>{html.escape(str(row.get(key, '')))}</td>" for key in keys) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>"))


def show_figures(figures, *, save_dir=None):
    """Display figures inline and optionally preserve vector PDF copies."""
    if save_dir is not None:
        Path(save_dir).mkdir(parents=True, exist_ok=True)
    for name, figure in figures.items():
        display(Markdown(f"**{name}**"))
        display(figure)
        if save_dir is not None:
            safe_name = "".join(character if character.isalnum() or character in "-_" else "_" for character in str(name))
            figure.savefig(Path(save_dir) / f"{safe_name}.pdf", bbox_inches="tight")
        plt.close(figure)


## Configure this campaign

Edit the paths before running. `weight_background_removed` already includes the observation weight; do not multiply it by `weight` again. With that default setting, the session can also construct the weight from `weight * background_removal_weight`.

Use a new `RUN_DIR` with `MODE="create"`. To continue a saved campaign, set `MODE="resume"`; the stored input paths and training configuration remain authoritative. `TRUTH_PATH=None` disables truth comparison. The truth file must be the full Herwig **signal** sample, not mixed and reconstruction-selected `data.root`.

The default network configuration matches BackgroundRemoval: four layers of 1024 swish neurons, four ensemble members, signed BCE, and the same calibration procedure. `device="cuda"` can require a GPU; `"auto"` allows CPU fallback.

`RatioConfig(inference_batch_size=...)` controls prediction and validation batches. Reduce it if GPU memory is limited. Saved ensembles reside on CPU between prediction calls and are moved to the selected device for evaluation.


In [ ]:
DATA_DIR = Path("/work/YOUR_GROUP/YOUR_USER/background_removed")
DATA_PATH = DATA_DIR / "data.root"
MC_PATH = Path("/work/YOUR_GROUP/YOUR_USER/merged/gg_H_pythia.root")
TRUTH_PATH = Path("/work/YOUR_GROUP/YOUR_USER/merged/gg_H_herwig.root")  # or None
RUN_DIR = Path("/work/YOUR_GROUP/YOUR_USER/omnifold_runs/campaign_001")
MODE = "create"  # change to "resume" after restarting Jupyter

MODEL_CONFIG = RatioConfig(device="auto")
DATA_WEIGHT_BRANCH = "weight_background_removed"
MC_WEIGHT_BRANCH = "weight"
SPLIT_SEED = 12345

# Resume can move a trained campaign between CPU and GPU without changing it.
RESUME_DEVICE = "auto"


In [ ]:
if MODE == "create":
    session = OmniFoldSession.create(
        data_path=DATA_PATH,
        mc_path=MC_PATH,
        run_dir=RUN_DIR,
        truth_path=TRUTH_PATH,
        config=MODEL_CONFIG,
        data_weight_branch=DATA_WEIGHT_BRANCH,
        mc_weight_branch=MC_WEIGHT_BRANCH,
        seed=SPLIT_SEED,
    )
elif MODE == "resume":
    session = OmniFoldSession.load(RUN_DIR, device=RESUME_DEVICE)
else:
    raise ValueError("MODE must be 'create' or 'resume'.")

show_rows(session.history())


## What one iteration does

For iteration $n$, let $T_{n-1}(z)$ be the previous cumulative truth ratio, with $T_0=1$.

| Model | Target | Reference | Inputs |
|---|---|---|---|
| Reco pull | Background-removed nominal data | MC with $w^0T_{n-1}$ | Reco |
| Truth push | MC with $w^0T_{n-1}R_n^{\rm pull}$ | Original MC with $w^0$ | Dressed truth |
| Reco likelihood | MC with $w^0T_n$ | Original MC with $w^0$ | Reco |

Every model stores its full ratio, normalized density ratio, and target/reference yield ratio separately. The truth push $T_n$ is already **cumulative**. The next iteration uses $w^0T_n$ directly. The third model's denominator remains the original MC at every iteration.

The reco training sample requires `fiducial && reconstructed && 115 < reco_m_ZZ < 130`. Truth training uses those same paired events and extends the fitted function to **all fiducial MC**, including reconstruction losses. This efficiency treatment assumes adequate response modeling and nonzero selected support in the truth region; it cannot measure invisible truth regions.

All three models use the same eight decay variables as BackgroundRemoval. Nominal data's truth fields are never used for fitting.

## Run the next iteration

**Rerun only this cell to append another iteration.** Each completed iteration is checkpointed. An interrupted fit can be repeated after reloading the campaign. There is no automatic stopping criterion.

In [ ]:
result = session.step()
print(f"Completed iteration {result.index}")
show_rows(session.history())


## Inspect the three density ratios

`ITERATION=None` selects the latest completed iteration. Set it to a completed iteration number to revisit earlier results. Diagnostics include signed-weight reweighting closure, raw/calibrated reliability, ensemble spread, and learning curves. Their closure partition is distinct from fit and calibration for each model, but the complete iterative procedure is adaptive; this is not a substitute for independent full-chain validation.

PDF copies can be saved beneath the campaign's `reports` directory.

In [ ]:
ITERATION = None
SAVE_PLOTS = True
iteration_label = ITERATION if ITERATION is not None else len(session.history())
report_dir = RUN_DIR / "reports" / f"iteration_{iteration_label:04d}"

ratio_figures = session.diagnostics(iteration=ITERATION)
show_figures(ratio_figures, save_dir=report_dir / "ratios" if SAVE_PLOTS else None)


## Compare reconstructed and fiducial distributions

The reconstructed comparison checks reweighted PYTHIA against background-removed data. The fiducial comparison checks the learned truth reweighting against the full Herwig signal. It includes fiducial events that did not pass reconstruction. All eight decay variables are shown.

Absolute expected-event yields are the default. Change `SHAPE_ONLY` to inspect shapes separately. A difference after background removal can include bias from the fixed signal-purity model, so interpret full-chain closure separately from the unfolding-only test.

In [ ]:
SHAPE_ONLY = False
comparison_figures = session.compare(iteration=ITERATION, shape_only=SHAPE_ONLY)
show_figures(comparison_figures, save_dir=report_dir / ("comparison_shapes" if SHAPE_ONLY else "comparison_yields") if SAVE_PLOTS else None)


## Track yields and evaluate the fixed-reference log ratios

The third ratio uses the original PYTHIA reco distribution as its reference at every iteration. The table exposes the evolution of the yield and weighted log-ratio objectives. A data event's observation weight remains its background-removed weight; the likelihood ratio does not become an extra observation weight.

With background-removal weights, these are **weighted likelihood objectives**. Their usual Poisson interpretation and asymptotic significance calibration are not automatic. The same nominal data trained this sequence, so the nominal objective curve is descriptive. For future parameter inference we must define and validate the statistical construction explicitly.

In [ ]:
show_rows(session.history())
show_rows(session.evaluate_likelihood())


The frozen sequence can also be evaluated on one other pseudo-experiment. This conditions on the nominal-data training and does not retrain either BackgroundRemoval or OmniFold for that toy.

In [ ]:
TOY_PATH = None  # e.g. DATA_DIR / "data_0001.root"
if TOY_PATH is not None:
    show_rows(session.evaluate_likelihood(data_path=TOY_PATH))


## Decorate the pseudo-experiment ensemble

Set `EXPORT_NOW=True` when ready, then run the cell. It writes new files in `EXPORT_DIR`, preserving the inputs. The vector position is the iteration index minus one.

| Branch stem | Stored sequence |
|---|---|
| `omnifold_reco_pull` | Incremental full reco ratios |
| `omnifold_truth_push` | Cumulative full truth ratios |
| `omnifold_reco_likelihood` | Cumulative full reco ratios against original MC |

Each stem also has `_shape` and `_yield_ratio` vectors and a scalar `_valid` mask. Yield factors are global and are repeated even on rows outside the evaluation domain. Data `omnifold_weight` entries repeat the unchanged background-subtracted observation weight; MC entries contain $w^0T_n$. Truth ratios on pseudo-data are diagnostic evaluations only when truth inputs exist. They neither identify a mixed event's component nor enter the data likelihood. For real data without truth, they are marked unavailable.

Use the same frozen models on all toys. The file list below includes nominal `data.root` and numbered `data_*.root` files.

In [ ]:
EXPORT_NOW = False
OVERWRITE_EXPORT = False  # only for replacing an existing recognized export
EXPORT_DIR = RUN_DIR / f"decorated_through_{len(session.history()):04d}"
export_inputs = sorted(DATA_DIR.glob("data*.root"))
print(f"Found {len(export_inputs)} data files for decoration.")
for path in export_inputs[:5]:
    print(path)
if len(export_inputs) > 5:
    print("...")

if EXPORT_NOW:
    if not export_inputs:
        raise FileNotFoundError(f"No data*.root files found in {DATA_DIR}")
    exported = session.decorate(inputs=export_inputs, output_dir=EXPORT_DIR, include_mc=True, overwrite=OVERWRITE_EXPORT)
    display(exported)
else:
    print("Set EXPORT_NOW=True to write the decorated files.")


## Continue later

Save the notebook, retain the campaign directory and unchanged input ROOT files, and start a new Unity allocation when needed. Choose the same kernel, set `MODE="resume"`, and run setup. Then inspect existing results or call `session.step()` again. Use a fresh campaign directory for changed input samples or model settings.

If a kernel or process was killed, `RUN_DIR/.training.lock` can remain. Confirm that the previous job and training process have stopped before removing a stale lock, then reload the campaign and call `session.step()` again. The last fully committed iteration is recovered; the interrupted iteration is fitted again. Abandoned `RUN_DIR/iterations/.iteration_*` staging directories are ignored on load and may be removed after training has stopped. Keep all committed `iteration_0001`, `iteration_0002`, etc. directories.
